## Plotly


Eine ganz andere Möglichkeit zur Visualisierung bietet [plotly](https://plotly.com/python).

Vorteile sind Interaktivität (Darstellung ist Javascript) und gute Defaults.

Wir nutzen nur ein Subset von Plotly: `plotly.express`.

### Visualisierungen für Datenanalyse

Visualisierungen erlauben es, ein Gefühl für Zusammenhänge in Daten zu bekommen.

In dieser intialen Phase wollen wir deshalb möglichst einfach verschiedene Ansichten durchprobieren.


Wir nutzen ein Beispieldatenset über die globale Entwicklung von Lebenserwartung und BIP, das mit plotly installiert wird. 

In [ ]:
import plotly.express as px

# Nur ein einzelnes Jahr.
df = px.data.gapminder().query('year == 2007')
df

In [ ]:
# Wie ist Lebenserwartung überhaupt verteilt?
px.strip(df, x='lifeExp')

In [ ]:
# Hover mit Ländernamen gibt ein erstes Gefühl
px.strip(df, x='lifeExp', hover_name='country')

In [ ]:
# Automatische Gruppierung nach einer Variable zeigt Zusammenhang 
# zwischen Kontinent und Lebenserwartung.
px.strip(df, x='lifeExp', hover_name='country', color='continent')

In [ ]:
# Bins werden automatisch eingeteilt, Höhe ist Anzahl der Länder.
px.histogram(df, x='lifeExp', hover_name='country', color='continent')

In [ ]:
# Marginal Plot zeigt zusätzlich die einzelnen Datenpunkte.
px.histogram(df, x='lifeExp', hover_name='country', color='continent', marginal="rug")

In [ ]:
# Mehr Sinn für die Höhe macht die Bevölkerung (pop).
px.histogram(
    df, 
    x='lifeExp', 
    y='pop',
    hover_name='country', 
    color='continent', 
    marginal="rug",
)

In [ ]:
# Subplots sind anhand einer Variable einfach.
px.histogram(
    df, 
    x='lifeExp', 
    y='pop',
    hover_name='country', 
    color='continent', 
    marginal='rug',
    facet_col='continent', 
)

In [ ]:
# Balken in einzelne Länder aufgeteilt, Lebenserwartung jetzt über die Farbe.
px.bar(
    df, 
    x='pop', 
    y='continent',
    hover_name='country', 
    color='lifeExp', 
    orientation='h',
)

In [ ]:
# Sunburst ist für Analysen als Teile eines Ganzen - Länder sind Teile eines Kontinents.
px.sunburst(
    df, 
    color='lifeExp', 
    values='pop',
    hover_name='country', 
    path=['continent', 'country']
)

In [ ]:
# Ähnlich Treemap
px.treemap(
    df, 
    color='lifeExp', 
    values='pop',
    hover_name='country', 
    path=['continent', 'country']
)

In [ ]:
# Einfache Karten:
px.choropleth(
    df, 
    color='lifeExp', 
    locations='iso_alpha',
    hover_name='country', 
)

#### Korrelationen überprüfen

Hypothese: Zusammenhang zwischen Wohlstand und Gesundheit?

Klassische Ansicht: Scatterplot.

In [ ]:
px.scatter(
    df, 
    x='gdpPercap', 
    y='lifeExp', 
    hover_name='country',
    color='continent',
    size='pop',
    )

In [ ]:
# Logarithmische x-Achse
px.scatter(
    df, 
    x='gdpPercap', 
    y='lifeExp', 
    hover_name='country',
    color='continent',
    size='pop',
    size_max=60,
    log_x=True,
    )

Diese Visualisierung ist durch den schwedischen Arzt Hans Rosling in TED Talks und anderen Medien berühmt geworden.

Das Datenset ist nach seiner Stiftung, der [Gapminder Foundation](https://www.gapminder.org/) benannt.

### Schönheitskorrekturen

Plots können auch in Plotly detailliert angepasst werden.

Der Return von z.B. `px.scatter` kann als Variable gespeichert und dann modifiziert werden.

In [ ]:
# Nicht verwechseln mit matplotlib fig!
fig = px.scatter(
    df, 
    x='gdpPercap', 
    y='lifeExp', 
    hover_name='country',
    color='continent',
    size='pop',
    size_max=60,
    log_x=True,
    )

Plotlys Figureobjekte sind intern JSON, das macht sie etwas leichter verständlich als matplotlib.

Alles im JSON kann auch noch verändert werden:

In [ ]:
print(fig.to_json(pretty=True))

In [ ]:
# Finales Design:
fig = px.scatter(
    df, 
    x='gdpPercap', 
    y='lifeExp', 
    hover_name='country',
    color='continent',
    size='pop',
    size_max=60,
    log_x=True,
    height=600,
    width=1000,
    # anderes Templates
    template='simple_white',
    color_discrete_sequence=px.colors.qualitative.G10,
    title="Health vs Wealth 2007",
    # Formatierte Label beim Hover
    labels={
        'continent': 'Continent',
        'pop': 'Population',
        'gdpPercap': 'GDP per Capita (US$ PPP)',
        'lifeExp': 'Life Expectancy (years)',
    },
)

# Zusätzliche Anpassungen
fig.update_layout(
    font_family='Rockwell',
    legend={
        'orientation': 'h',
        'title': '',
        'y': 1.1,
        'x': 1,
        'xanchor': 'right',
        'yanchor': 'bottom',
    }
)
# Feste Definition der Achsen.
fig.update_xaxes(tickprefix='$', range=[2, 5], dtick=1)
fig.update_yaxes(range=[30, 90])
# Zusätzliche Linien zeigen gewichtete Durchschnitte.
fig.add_hline((df['lifeExp']*df['pop']).sum() / df['pop'].sum(), line_width=1, line_dash='dot')
fig.add_vline((df['gdpPercap']*df['pop']).sum() / df['pop'].sum(), line_width=1, line_dash='dot')
# Wird erst mit fig.show() angezeigt.
fig.show()

### Übung

Reproduziere den Plot mit den Fluggastzahlen 2020 aus dem Matplotlib-Teil.

Hinweis:
- Nutze px.line für einen Linienplot.

Daten sind gegeben:

In [ ]:
import plotly.express as px
import pandas as pd

tsa_melted_holiday_travel = pd.read_csv(
    '../data/tsa_melted_holiday_travel.csv', 
    parse_dates=True, index_col='date'
)
tsa_2020 = tsa_melted_holiday_travel.loc['2020'].copy().drop(columns=['year'])
tsa_2020['7D mean'] = tsa_2020.rolling('7D').travelers.mean()
tsa_2020['YTD mean'] = tsa_2020.expanding().travelers.mean()

# Lösung hier...

.

.

.

.

.

.

.

.

.

.

.

.

In [ ]:
fig = px.line(tsa_2020, y=['travelers', '7D mean', 'YTD mean'])
fig.update_layout({"xaxis_title": "Date", "yaxis_title": "Travelers"})
fig.show()